# Hugging Face to Spark NLP: OpenVINO Conversion Guide

This notebook walks you through converting any compatible Hugging Face model to a Spark NLP compatible format using OpenVINO.
What this covers:
- Exporting a Hugging Face model to OpenVINO IR format using optimum-intel
- Preparing model assets (vocab, labels, tokenizer files)
- Loading the exported model into Spark NLP
- Running an end-to-end inference pipeline

# openvino setup/conversion

In [ ]:
!python -m pip install -q "optimum-intel[openvino]"@git+https://github.com/huggingface/optimum-intel.git

Name of the model you want to convert from Hugging Face: https://huggingface.co/models

In [ ]:
model_id = "FacebookAI/roberta-large-mnli"

In [ ]:
!optimum-cli export openvino \
  --model $model_id \
  --task text-classification \
  --trust-remote-code \
  openvino

In [ ]:
!ls openvino

run for more info: `!optimum-cli export openvino --help`

In [ ]:
# @title assets prep function
# This function prepares the assets for an OpenVINO exported model.
# It does the following:
# - Creates an `assets` folder inside the model directory.
# - Moves all files except `.xml` and `.bin` files into `assets`.
# - Generates `vocab.txt` from `vocab.json` or `tokenizer.json` if missing.
# - Generates `labels.txt` from `config.json` if the model is for classification.
# - Warns if `merges.txt` is missing (required for BPE tokenizers like RoBERTa/GPT2).
# - If no tokenizer files exist, it can download them from HuggingFace using the provided model ID.

import os, json, re, shutil, time, requests
from pathlib import Path
from transformers import AutoTokenizer
from google.colab import files, userdata

def prepare_assets(
    model_path: str = "openvino",
    classification: bool = False,
    which_tokenizer: str = None
):
    model_path = Path(model_path)
    assets_path = model_path / "assets"
    os.makedirs(assets_path, exist_ok=True)

    # Check specifically for tokenizer-related files, not just any .json/.txt
    # (avoids false positives from config.json or other OV export artifacts)
    has_assets = (
        (assets_path / "vocab.json").exists() or
        (assets_path / "tokenizer.json").exists() or
        (assets_path / "vocab.txt").exists() or
        (model_path / "vocab.json").exists() or
        (model_path / "tokenizer.json").exists() or
        (model_path / "vocab.txt").exists()
    )

    if not has_assets and which_tokenizer:
        print(f"No tokenizer assets found in {model_path}, downloading tokenizer '{which_tokenizer}'...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(which_tokenizer, use_fast=False)
            tokenizer.save_pretrained(assets_path)
            print(f"Tokenizer saved to {assets_path}")
        except Exception as e:
            print(f"Failed to download tokenizer: {e}")
            return

    # Move all non-.xml/.bin files into assets/
    for file_path in list(model_path.iterdir()):
        if file_path.is_file() and file_path.suffix not in (".xml", ".bin"):
            shutil.move(str(file_path), assets_path / file_path.name)

    # Generate vocab.txt if missing
    vocab_txt = assets_path / "vocab.txt"
    if not vocab_txt.exists():
        vocab_json = assets_path / "vocab.json"
        tokenizer_json = assets_path / "tokenizer.json"

        if vocab_json.exists():
            try:
                with open(vocab_json, "r", encoding="utf-8") as f, open(vocab_txt, "w", encoding="utf-8") as out:
                    vocab = json.load(f)
                    for token in vocab.keys():
                        out.write(token + "\n")
                print("Generated vocab.txt from vocab.json")
            except Exception as e:
                print(f"Failed to generate vocab.txt from vocab.json: {e}")

        elif tokenizer_json.exists():
            try:
                with open(tokenizer_json, "r", encoding="utf-8") as f, open(vocab_txt, "w", encoding="utf-8") as out:
                    vocab = json.load(f).get("model", {}).get("vocab", [])
                    for e in vocab:
                        token = (
                            e if isinstance(e, str)
                            else e[0] if isinstance(e, (list, tuple)) and e
                            else e.get("token") if isinstance(e, dict)
                            else None
                        )
                        if token is not None:
                            out.write(token + "\n")
                print("Generated vocab.txt from tokenizer.json")
            except Exception as e:
                print(f"Failed to generate vocab.txt from tokenizer.json: {e}")

    # Warn if merges.txt is missing (required for BPE tokenizers: RoBERTa, GPT-2, etc.)
    merges_txt = assets_path / "merges.txt"
    if not merges_txt.exists():
        vocab_json = assets_path / "vocab.json"
        # BPE tokenizers use vocab.json + merges.txt together; warn if vocab.json is present
        if vocab_json.exists():
            print("Warning: merges.txt not found but vocab.json exists — if this is a BPE tokenizer "
                  "(RoBERTa, GPT-2, etc.), the tokenizer will likely fail at runtime. "
                  "Try setting which_tokenizer to the base model ID to re-download assets.")
        else:
            print("Warning: merges.txt not found. If your model uses a BPE tokenizer, this may cause issues.")

    # Generate labels.txt for classification models
    if classification:
        config_json = assets_path / "config.json"
        labels_txt = assets_path / "labels.txt"
        if config_json.exists() and not labels_txt.exists():
            try:
                with open(config_json, "r", encoding="utf-8") as f:
                    labels = json.load(f).get("id2label", {}).values()
                with open(labels_txt, "w", encoding="utf-8") as f:
                    f.write("\n".join(labels))
                print("Generated labels.txt from config.json")
            except Exception as e:
                print(f"Failed to generate labels.txt: {e}")

    # Final summary of what's in assets/
    asset_files = [f.name for f in assets_path.iterdir() if f.is_file()]
    print(f"\nAssets preparation complete at {assets_path}")
    print(f"Files in assets/: {asset_files}")


In [ ]:
prepare_assets(
    model_path="openvino",
    classification=True,
)

In [ ]:
# we will explicitly need to download merges.txt
!wget -q https://huggingface.co/FacebookAI/roberta-large-mnli/resolve/main/merges.txt -O openvino/assets/merges.txt


In [ ]:
!ls openvino/assets

# import in sparknlp

In [ ]:
# This is only to setup PySpark and Spark NLP on Colab
!wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [ ]:
import sparknlp

spark = sparknlp.start()

print("Spark NLP version: ", sparknlp.version())
print("Apache Spark version: ", spark.version)

specify the name of the model you want in sparknlp here

In [ ]:
sparknlp_model_name = "roberta-large-mnlim"

Dont forget `.setStorageRef()` here if its an embedding - it should be the same as the model name so

`.setStorageRef(sparknlp_model_name)`

In [ ]:
from sparknlp.annotator import RoBertaForSequenceClassification

imported_model = (
    RoBertaForSequenceClassification.loadSavedModel("openvino", spark)
      .setInputCols(["token", "document"])
      .setOutputCol("answer")
)


In [ ]:
print(imported_model.explainParams())

In [ ]:
imported_model.write().overwrite().save(sparknlp_model_name)

In [ ]:
!ls -lh $sparknlp_model_name

test imported model

In [ ]:
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

documentAssembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

loaded_model = RoBertaForSequenceClassification \
    .loadSavedModel("openvino", spark) \
    .setInputCols(["document", "token"]) \
    .setOutputCol("label") \
    .setCaseSensitive(True)

pipeline = Pipeline().setStages([
    documentAssembler,
    tokenizer,
    loaded_model
])

data = spark.createDataFrame(
    [
        ("I loved this movie when I was a child.",),
        ("It was pretty boring.",)
    ],
    ["text"]
)

result = pipeline.fit(data).transform(data)
result.select("text", "label.result").show(truncate=False)
